# 11 · SGD linear regression (ML regression)

Fit the binary label with `SGDRegressor(loss="squared_error")` on the same 84 cached tabular features as notebook 10. The grid explores regularisation, initial learning rate, and learning-rate schedule.

**Data boundary:** frozen train and validation tabular archives only. No audio, mel cache, test manifest, or test feature archive is read. Thresholds and selection use validation only.


In [ ]:
from pathlib import Path
import sys, json, time, warnings, itertools
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import SGDRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from dronedet.config import load_project_config
from dronedet.features import load_tabular
from dronedet.metrics import classification_metrics, select_f1_threshold, miss_rate_at_false_alarm_rate

cfg = load_project_config(ROOT / 'config.yaml')
P = cfg['P']


## Frozen inputs and scaler policy

Q-A selects each model by maximum validation F1, choosing the higher threshold on ties; ROC-AUC and miss rate at 1%/5% false-alarm caps are reported. Q-D fits the scaler on train rows excluding `silent` and `near_silent` only. `crop_all_zero` alone does not exclude a row.


In [ ]:
train = load_tabular(P, 'train')
val = load_tabular(P, 'val')
X_train = np.asarray(train['X'], dtype=np.float64)
y_train = np.asarray(train['y'], dtype=np.int8)
X_val = np.asarray(val['X'], dtype=np.float64)
y_val = np.asarray(val['y'], dtype=np.int8)
feature_names = np.asarray(train['feature_names']).astype(str)
assert X_train.shape == (7607, 84) and X_val.shape == (1510, 84)
assert np.isfinite(X_train).all() and np.isfinite(X_val).all()
assert np.array_equal(train['clip_id'], np.asarray(pd.read_csv(P['splits'] / 'train.csv', keep_default_na=False).clip_id))
assert np.array_equal(val['clip_id'], np.asarray(pd.read_csv(P['splits'] / 'val.csv', keep_default_na=False).clip_id))

excluded = np.asarray(train['silent'], bool) | np.asarray(train['near_silent'], bool)
fit_rows = ~excluded
assert int(fit_rows.sum()) == 7557
scaler = StandardScaler().fit(X_train[fit_rows])
Xtr = scaler.transform(X_train)
Xva = scaler.transform(X_val)
sample_weight = compute_sample_weight(class_weight='balanced', y=y_train)
print('Train/validation rows:', len(y_train), len(y_val))
print('Scaler-fit rows:', int(fit_rows.sum()), 'of', len(fit_rows))


## Default and prewritten grid

The default is `alpha=1e-4`, `eta0=1e-2`, `learning_rate="invscaling"`, L2, `max_iter=1000`, `tol=1e-3`, and `early_stopping=False`. The 30 configurations are evaluated over three fixed seeds. A convergence warning triggers a logged retry at 10,000 iterations.


In [ ]:
alphas = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1]
eta0s = [1e-3, 1e-2, 1e-1]
schedules = ['invscaling', 'adaptive']
seeds = [int(cfg['seed']), int(cfg['seed']) + 1, int(cfg['seed']) + 2]
configs = [dict(alpha=a, eta0=e, learning_rate=l) for a,e,l in itertools.product(alphas, eta0s, schedules)]
default = dict(alpha=1e-4, eta0=1e-2, learning_rate='invscaling')
records=[]
models={}
t0=time.perf_counter()
for ci, hp in enumerate(configs):
    for seed in seeds:
        def fit_model(max_iter):
            return SGDRegressor(loss='squared_error', penalty='l2', alpha=hp['alpha'], eta0=hp['eta0'],
                                learning_rate=hp['learning_rate'], max_iter=max_iter, tol=1e-3,
                                early_stopping=False, random_state=seed, shuffle=True)
        model=fit_model(1000)
        caught=[]
        with warnings.catch_warnings(record=True) as ws:
            warnings.simplefilter('always', ConvergenceWarning)
            model.fit(Xtr, y_train, sample_weight=sample_weight)
            caught=[str(w.message) for w in ws if issubclass(w.category, ConvergenceWarning)]
        retry=False
        if caught:
            retry=True
            model=fit_model(10000)
            with warnings.catch_warnings(record=True) as ws:
                warnings.simplefilter('always', ConvergenceWarning)
                model.fit(Xtr, y_train, sample_weight=sample_weight)
                caught += [str(w.message) for w in ws if issubclass(w.category, ConvergenceWarning)]
        scores=model.predict(Xva)
        threshold=select_f1_threshold(y_val, scores)
        metrics=classification_metrics(y_val, scores, threshold=threshold)
        row={'config_id':ci,'seed':seed,**hp,'threshold':float(threshold),'val_f1':metrics['f1'],
             'val_roc_auc':metrics['roc_auc'],'false_alarm_rate':metrics['false_alarm_rate'],
             'miss_rate':metrics['miss_rate'],'n_iter':int(model.n_iter_),'retry_10000':retry,
             'convergence_warning_after_retry':bool(caught)}
        records.append(row); models[(ci,seed)]=model
results=pd.DataFrame(records)
print('Fits:',len(results),'| elapsed seconds:',round(time.perf_counter()-t0,2))
print('Retries:',int(results.retry_10000.sum()),'| warnings remain after retry:',int(results.convergence_warning_after_retry.sum()))


## Validation selection and baseline replacement rule

Select the configuration with highest mean validation F1 across seeds (ties resolve toward the default, then lower alpha and eta0). Keep a non-default only when its mean F1 exceeds the default by more than the default seed standard deviation and by at least 0.01. Otherwise freeze the default.


In [ ]:
summary = results.groupby(['config_id','alpha','eta0','learning_rate'],as_index=False).agg(
    mean_val_f1=('val_f1','mean'), std_val_f1=('val_f1','std'), mean_roc_auc=('val_roc_auc','mean'),
    mean_far=('false_alarm_rate','mean'), mean_miss=('miss_rate','mean'), mean_n_iter=('n_iter','mean'),
    retried=('retry_10000','sum'), warnings_after_retry=('convergence_warning_after_retry','sum'))
def is_default(r): return r.alpha==default['alpha'] and r.eta0==default['eta0'] and r.learning_rate==default['learning_rate']
summary['is_default']=summary.apply(is_default,axis=1)
default_row=summary.loc[summary.is_default].iloc[0]
summary['default_delta_f1']=summary.mean_val_f1-float(default_row.mean_val_f1)
summary['passes_replace_rule']=(~summary.is_default) & (summary.default_delta_f1 >= 0.01) & (summary.default_delta_f1 > float(default_row.std_val_f1))
ranked=summary.sort_values(['mean_val_f1','is_default','alpha','eta0'],ascending=[False,False,True,True])
best_candidate=ranked.iloc[0]
chosen=best_candidate if bool(best_candidate.is_default) or bool(best_candidate.passes_replace_rule) else default_row
chosen_id=int(chosen.config_id)
seed_rows=results[results.config_id==chosen_id].sort_values('seed')
chosen_seed=seed_rows.sort_values(['val_f1','seed'],ascending=[False,True]).iloc[0]
chosen_model=models[(chosen_id,int(chosen_seed.seed))]
chosen_scores=chosen_model.predict(Xva)
threshold=select_f1_threshold(y_val,chosen_scores)
metrics=classification_metrics(y_val,chosen_scores,threshold=threshold)
miss1,thr1=miss_rate_at_false_alarm_rate(y_val,chosen_scores,max_false_alarm_rate=.01)
miss5,thr5=miss_rate_at_false_alarm_rate(y_val,chosen_scores,max_false_alarm_rate=.05)
print('Default mean F1/std:',float(default_row.mean_val_f1),float(default_row.std_val_f1))
print('Top candidate:',best_candidate[['alpha','eta0','learning_rate','mean_val_f1']].to_dict())
print('Frozen config:',{k:chosen[k] for k in ['alpha','eta0','learning_rate']},'| selected seed:',int(chosen_seed.seed))
print('Chosen seed validation metrics:',metrics)


## Diagnostics and freeze

The plot shows validation F1 across the prewritten grid; the coefficient comparison reports distance from notebook 10's least-squares coefficients. The frozen model is the best seed of the selected configuration by validation F1. No test data is opened.


In [ ]:
FIG=ROOT/'results'/'figures'; TABLES=ROOT/'results'/'tables'; MODELS=ROOT/'results'/'models'; CARDS=ROOT/'results'/'cards'
for d in (FIG,TABLES,MODELS,CARDS): d.mkdir(parents=True,exist_ok=True)
summary.to_csv(TABLES/'11_sgd_regression_grid_summary.csv',index=False)
results.to_csv(TABLES/'11_sgd_regression_seed_metrics.csv',index=False)
fig,ax=plt.subplots(figsize=(9,5))
for sched,g in results.groupby('learning_rate'):
    ax.scatter(g.alpha.astype(str)+' / '+g.eta0.astype(str),g.val_f1,label=sched,s=18,alpha=.65)
ax.set(xlabel='alpha / eta0',ylabel='Validation F1',title='SGD regression grid (each seed)'); ax.tick_params(axis='x',rotation=90); ax.legend(); fig.tight_layout()
fig.savefig(FIG/'11_sgd_regression_grid.png',dpi=160); plt.close(fig)
# Compare coefficients with notebook 10, when that frozen artifact exists.
lin_path=MODELS/'10_linear_regression.joblib'
coef_distance=None; auc_delta_vs_10=None
if lin_path.exists():
    lin_coef=pd.read_csv(TABLES/'10_linear_regression_coefficients.csv').set_index('feature').loc[feature_names].coefficient.to_numpy()
    coef_distance=float(np.linalg.norm(chosen_model.coef_-lin_coef))
    auc_delta_vs_10=float(metrics['roc_auc']-json.loads((CARDS/'10_linear_regression.json').read_text(encoding='utf-8'))['val_metrics']['roc_auc'])
front_hash=json.loads((P['spec']/'frontend_hash.json').read_text(encoding='utf-8'))['params_hash']
model_path=MODELS/'11_sgd_regression.joblib'
joblib.dump({'model':chosen_model,'scaler':scaler,'feature_names':feature_names.tolist(),'threshold':float(threshold),
 'threshold_rule':'validation_f1_max_ties_higher','scaler_fit_exclusion':'silent OR near_silent; crop_all_zero alone remains eligible',
 'front_end_hash':front_hash,'hyperparameters':{k:chosen[k].item() if hasattr(chosen[k],'item') else chosen[k] for k in ['alpha','eta0','learning_rate']},
 'seed':int(chosen_seed.seed)},model_path)
card={'model':'sgd_regression','representation':'84 tabular features','best_hyperparameters':{k:chosen[k].item() if hasattr(chosen[k],'item') else chosen[k] for k in ['alpha','eta0','learning_rate']},
 'val_metrics':{**metrics,'threshold':float(threshold),'miss_rate_at_far_1pct':float(miss1),'threshold_at_far_1pct':float(thr1),
 'miss_rate_at_far_5pct':float(miss5),'threshold_at_far_5pct':float(thr5),'mean_f1_across_seeds':float(chosen.mean_val_f1),'std_f1_across_seeds':float(chosen.std_val_f1)},
 'seed_metrics':seed_rows.to_dict(orient='records'),'seeds':seeds,'train_time_s':float(time.perf_counter()-t0),
 'diagnostics':{'coefficient_l2_distance_vs_notebook10':coef_distance,'roc_auc_delta_vs_notebook10':auc_delta_vs_10},
 'convergence_retries':int(chosen.retried),'warnings_after_retry':int(chosen.warnings_after_retry),
 'decision_log_ids':['D044','D045','D047'],'model_path':str(model_path)}
(CARDS/'11_sgd_regression.json').write_text(json.dumps(card,indent=2,default=float),encoding='utf-8')
print('Saved model and validation verdict card.')
